In [7]:
!pip install -q pydantic fastapi

# Basics, I/O

In [ ]:
from pydantic import BaseModel, Field
from typing import Annotated, Union, Any
from fastapi import FastAPI, Query, Body, Cookie, Header, status, HTTPException, Depends
from enum import Enum

app = FastAPI()


app.r

class Tags(Enum):
    items = "items"
    users = "users"

class APIInput(BaseModel):
    model_config = {"extra": "forbid"}
    message: str = Field(description="User input")

class APIOutput(BaseModel):
    message: str

class Cookies(BaseModel):
    model_config = {"extra": "forbid"}

    session_id: str
    fatebook_tracker: Union[str, None] = None
    googall_tracker: Union[str, None] = None

# Variable to be annotated as Query() for GET requests in the URL string
# Variable to be annotated as Body() for PUT/POST/etc. in the request payload
# Path is to validate the URL path
# Similarly for Header, Cookie, Form and File

@app.get("/", tags = [Tags.users])
async def root(input_message:APIInput):
    return APIOutput({"message":input_message})

@app.get("/joke", tags=[Tags.items])
async def tell_joke(input_message: Annotated[str|None, Query()]=None):
    return APIOutput({"message":"No joke"})

@app.put("/items/{item_id}", status_code=status.HTTP_200_OK, tags=["Item"])
async def update_item(item_id: int, item: APIInput):
    results = {"item_id": item_id, "item": item}
    return results

@app.get("/err")
async def err():
    raise HTTPException(status_code=404, detail="Random error message")


@app.put("/items/{item_id}", response_model=dict[str, float], status_code=status.HTTP_202_ACCEPTED)
async def update_item(
    item_id: int,
    item: Annotated[
        APIInput,
        Body(
            examples=[
                {
                    "name": "Foo",
                    "description": "A very nice Item",
                    "price": 35.4,
                    "tax": 3.2,
                }
            ],
        ),
    ],
):
    results = {"item_id": item_id, "item": item}
    return results


@app.get("/items/")
async def read_items(cookies: Annotated[Cookies, Cookie()]):
    return cookies

# Dependency injection

In [ ]:
# The simplicity of the dependency injection system makes FastAPI compatible with:

# * all the relational databases
# * NoSQL databases
# * external packages
# * external APIs
# * authentication and authorization systems
# * API usage monitoring systems
# * response data injection systems

# Anything that can be "called" can be used as a dependency, a function or an object even

async def common_parameters(q: str | None = None, skip: int = 0, limit: int = 100):
    return {"q": q, "skip": skip, "limit": limit}


CommonsDep = Annotated[dict, Depends(common_parameters)]


@app.get("/items/")
async def read_items(commons: CommonsDep):
    return commons


@app.get("/users/")
async def read_users(commons: CommonsDep):
    return commons


################################################################################


class CommonQueryParams:
    def __init__(self, q: str | None = None, skip: int = 0, limit: int = 100):
        self.q = q
        self.skip = skip
        self.limit = limit


@app.get("/items/")
async def read_items(commons: Annotated[Any, Depends(CommonQueryParams)]):
    response = {}

################################################################################


# In 
commons: Annotated[CommonQueryParams, Depends(CommonQueryParams)]
# The first CommonQueryParams is useless as fastAPI refers to the Depends dependency
# It can be replaced with Any



# Dependency can be chained: a depends on b; b depends on c


# Depends can be used with contextlib.asyncontextmanager or contextlib.contextmanager & yield

# Security

In [ ]:
# Authorization+PKCE OAuth2 workflow

# OAuth2 Google authentication

# Required things

# 1. Third party user authenticator(Google/Facebook etc.)
    # We need to register our server & generate the server's client ID and client secret
# 2. Callback URL in the server after OAuth2 authentication

# OAuth2 workflow:

# Client browser - [Server client ID + state + scope + state + redirect URI] -> Third party authenticator(Google/FB) -> User login page for Google/FB -> Auth Code -> Callback URL in server
# Server callback URL - [Auth Code] -> Third party token URL -> Access token -> Server generates JWT token with expiry -> Client browser
# -> This access token is sent in every request to the server.

# /login : Redirects to third party login
# /callback : Handles exchanging of access token for authorization token
# /token : Generates the token
# /session : Creates JWT token and returns user object/session

<img src="IMG_6412.jpeg" width="50%" />

# Middleware


In [4]:

# Interecepts incoming & outgoing requests

# If multiple middlewares are added, they are stacked in FIFO format with the first middleware being the innermost
# Middlewares can be used to add custom headers

# CORS

In [6]:
# When frontend & backend are in different origins(always in prod scenarios)

# Cross Origin Resource Sharing defines:
# 1. Allowed domains from which incoming is allowed 
# 2. Allowed method requests to backend
# 3. Whether headers are allowed
# 4. Whether cookies are allowed(credentials)

# Structuring bigger applications

In [ ]:
# moduleB

from fastapi import APIRouter, Header
router = APIRouter(tags=["admin"])

@router.get("/login")
def login(x_token: Annotated[str, Header()]):
    pass

In [ ]:
# moduleA

from fastapi import APIRouter
from moduleB import login
router = APIRouter(
    prefix='/items',
    tags=["items"],
    dependencies=[Depends(login)],
    responses={200:{"description":"okay"},
               400:{"description":"err"},
               403:{"description":"forbidden"}})

@router.get("/a1/{item}")
async def s1(item: Annotated[str, "Item input"]):
    return {"item": item}

In [ ]:
# main

from fastapi import APIRouter
from moduleA import router as routerA
from moduleB import router as routerB

router = APIRouter()
router.include_router(routerB)
router.include_router(routerA)

# Output formats

In [11]:
# Return as JSON or pydantic objects
# ORJSON is the format if speed is the priority

# Output can also be a HTML file or a streamable

# Advanced topics

In [ ]:
# 1. JSON, HTML and streamable outputs
# 2. Returning response headers, cookies; changing response status code
# 3. Websockets - very different design -> Good resource management and application design

In [ ]:
# 4. Lifespan events - startup & shutdown code; yield vs return
# 5. Context manager - sync vs async context manager

# Synchronous context example
# with open(...) as f:
    # f.read()

# Async context is used in fastAPI application where resources need to be awaited

############################################################################################################################
###############################                            IMPORTANT                            ############################
############################################################################################################################


# 6. Event loops, Task, Coroutines, No. of workers

* No. of CPU cores = no. of really parallel CPU bound possible. However, OS schedules threads into cores.
* When the app is deployed, `n` workers are each start an event loop with the app. Each of these workers individually cater to incoming requests.
* Each worker has its own memory, python interpreter and event loop.
* Rule of thumb is no. of workers = no. of cores. Python has GIL, hence a worker/process cannot use more than one core for CPU bound work.

Event loop is a smart scheduler that runs thousands of lightweight tasks on a single thread

* Each function/API that is awaited is a co-routine. When it is awaited, a Task is created.
* The event loop exeutes a Task when its ready till it is finished or awaits something(DB connection, I/O etc.), when it resumes another ready Task
* The previous process signals ready when its awaiting is done

No thread switching -> Very fast execution


In [ ]:
# 7. Failue, resilience and load management

# Think in layers:

# Traffic control & load protection
#   ├─ Rate Limiting / Throttling
#   ├─ Load shedding
#   └─ Backpressure

# Failure isolation & resilience
#   ├─ Circuit breaker
#   ├─ Bulkhead pattern
#   ├─ Timeout
#   └─ Exponential backoff with retries

# Concurrency & Resource Management
#   ├─ Concurrency limiting
#   ├─ Request queuing
#   └─ Connection pool limits

# Consistency & safety patterns
#   ├─ Idemnpotency
#   └─ Request de-duplication

# Observability driven control
#   ├─ Adaptive rate-limiting
#   └─ Health-checks

# Security & Abuse Prevention
#   ├─ Quota management
#   └─ Request validation & size limits



# ## 1. Traffic Control & Load Protection
# These protect your service from being overwhelmed.

# ### 1. Rate Limiting
# Limit requests per user/IP/token.
# * Fixed window, sliding window, token bucket, leaky bucket
# * FastAPI tools: `slowapi`, `fastapi-limiter`, API gateway (NGINX, Kong)

# ### 2. **Throttling**
# * Gradually slows clients instead of hard-rejecting
# * Often used in APIs with different tiers (free vs paid)

# 👉 Rate limiting is *binary*, throttling is *progressive*.

# ### 3. **Load Shedding**
# Intentionally dropping requests when overloaded.
# * Return `503 Service Unavailable`
# * Can be random, priority-based, or queue-length-based
# 👉 Better to drop traffic than crash the service.

# ### 4. **Backpressure**

# Signal clients or upstream services to slow down.
# * Queue limits
# * Bounded async queues
# * HTTP 429 or 503 responses

# 👉 Very relevant in async FastAPI apps with message queues or streaming.

# ---

# ## 2. Failure Isolation & Resilience
# These stop failures from cascading across services.

# ### 5. Circuit Breaker
# Stops calling a failing dependency.
# * States: **Closed → Open → Half-Open**
# * Prevents resource exhaustion
# 👉 Especially important for DBs, external APIs, LLM calls.

# ### 6. **Bulkhead Pattern**
# Isolate resources so one failure doesn’t sink everything.
# * Separate thread pools / event loops
# * Separate DB connection pools per feature
# 👉 Example: Search API going down shouldn’t kill Auth API.

# ### 7. **Timeouts**
# Fail fast instead of waiting forever.
# * HTTP client timeouts
# * DB query timeouts
# 👉 Always combine with circuit breakers.

# ### 8. **Retry (with backoff & jitter)**
# Retry transient failures safely.
# * Exponential backoff
# * Random jitter to avoid thundering herd
# 👉 Never retry blindly—combine with circuit breakers.

# ---

# ## 3. Concurrency & Resource Management
# Control *how much work* your app does in parallel.

# ### 9. **Concurrency Limiting**
# Limit simultaneous requests or tasks.
# * Semaphore-based limits
# * Per-endpoint concurrency caps
# 👉 Crucial for CPU-heavy ML inference endpoints.

# ### 10. **Request Queuing**
# Queue requests instead of rejecting immediately.
# * In-memory queue
# * External queues (Redis, RabbitMQ, Kafka)
# 👉 Trade-off: latency vs availability.


# ### 11. **Connection Pooling Limits**
# Limit DB or external service connections.
# * SQLAlchemy pool size
# * HTTP client pool limits
# 👉 Prevents exhausting downstream systems.

# ---

# ## 4. Consistency & Safety Patterns
# Protect correctness under load.

# ### 12. **Idempotency**
# Safe retries without side effects.
# * Idempotency keys
# * Deduplication in Redis
# 👉 Critical for payment-like APIs.

# ### 13. **Request Deduplication**
# Detect and collapse identical requests.
# * Cache in-flight requests
# * Especially useful for expensive computations

# ---

# ## 5. Observability-Driven Control
# Patterns that react based on metrics.

# ### 14. **Adaptive Rate Limiting**
# Rate limits change based on:
# * CPU usage
# * Error rate
# * Latency
# 👉 Smarter than static limits.


# ### 15. **Health Checks & Readiness Probes**
# Expose service state.
# * Liveness
# * Readiness (stop traffic when unhealthy)

# 👉 Kubernetes relies heavily on this.

# ---

# ## 6. Security & Abuse Prevention (Adjacent)
# Often implemented alongside rate limiters.

# ### 16. **Quota Management**
# Long-term usage limits.
# * Daily/monthly request caps
# * Per-user or per-tenant quotas

# ### 17. **Request Validation & Size Limits**
# Prevent pathological requests.
# * Max body size
# * Max query params
# * Schema validation


# ## FastAPI-Specific Practical Stack (Recommended)

# For a **production-grade FastAPI app**, a solid baseline is:

# * ✅ Rate Limiter (Redis-based)
# * ✅ Per-endpoint concurrency limits
# * ✅ Timeouts on *every* external call
# * ✅ Retry + exponential backoff
# * ✅ Circuit breaker for unstable dependencies
# * ✅ Health/readiness endpoints
# * ✅ Metrics (Prometheus) → adaptive controls later

In [ ]:
# 8. Pydantic Settings object

# Clean way to build a tree based and dependency injection supported config creation

# Obtain the settings from cache using @lru_cache decorator